## Person C

###Step-1

####code_runner.py

In [ ]:

import subprocess
import tempfile
import sys
import os


def run_student_code(student_code, test_code, timeout=3):

    # Create a temporary Python file
    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".py",
        delete=False,
        encoding="utf-8"
    ) as file:

        file.write(student_code)
        file.write("\n\n")
        file.write(test_code)

        file_path = file.name

    try:

        # Execute the student's code
        result = subprocess.run(
            [sys.executable, file_path],
            capture_output=True,
            text=True,
            timeout=timeout
        )

        # Check whether the tests passed
        if result.returncode == 0:

            return {
                "passed": True,
                "error_type": None,
                "output": result.stdout,
                "traceback": None
            }

        else:

            error_line = result.stderr.strip().splitlines()

            return {
                "passed": False,
                "error_type": (
                    "ASSERTION_ERROR"
                     if "AssertionError" in result.stderr
                     else "RUNTIME_ERROR"
                  ),
                "output": result.stdout,
                "traceback": result.stderr
            }

    except subprocess.TimeoutExpired:

        return {
            "passed": False,
            "error_type": "TIMEOUT",
            "output": "",
            "traceback": "Code execution exceeded the time limit."
        }

    finally:

        # Delete the temporary file
        if os.path.exists(file_path):
            os.remove(file_path)


####test_runner.py

In [ ]:
#from code_runner import run_student_code


# TEST 1: Correct code

student_code = """
def add(a, b):
    return a + b
"""

test_code = """
assert add(5, 3) == 8
assert add(10, 20) == 30

print("All tests passed!")
"""

result = run_student_code(student_code, test_code)

print("TEST 1 RESULT:")
print(result)


# TEST 2: Incorrect logic

student_code = """
def add(a, b):
    return a - b
"""

result = run_student_code(student_code, test_code)

print("\nTEST 2 RESULT:")
print(result)


# TEST 3: Infinite loop

student_code = """
while True:
    pass
"""

test_code = """
print("Testing")
"""

result = run_student_code(student_code, test_code)

print("\nTEST 3 RESULT:")
print(result)


TEST 1 RESULT:
{'passed': True, 'error_type': None, 'output': 'All tests passed!\n', 'traceback': None}

TEST 2 RESULT:
{'passed': False, 'error_type': 'ASSERTION_ERROR', 'output': '', 'traceback': 'Traceback (most recent call last):\n  File "/tmp/tmpkgdhein2.py", line 7, in <module>\n    assert add(5, 3) == 8\n           ^^^^^^^^^^^^^^\nAssertionError\n'}

TEST 3 RESULT:
{'passed': False, 'error_type': 'TIMEOUT', 'output': '', 'traceback': 'Code execution exceeded the time limit.'}


###Step-2

In [9]:
def diagnose_error(runner_result, skill="SKL-02"):

    error_type = runner_result.get("error_type")
    traceback = runner_result.get("traceback") or ""

    if error_type == "TIMEOUT":
        misconception_id = "MIS-04"

    elif "NoneType" in traceback:
        misconception_id = "MIS-02"

    elif error_type in ["ASSERTION_ERROR", "RUNTIME_ERROR"]:
        misconception_id = "MC_UNKNOWN"

    else:
        misconception_id = None

    if misconception_id not in [None, "MC_UNKNOWN"]:

        misconception = next(
            item for item in domain_data["misconceptions"]
            if item["id"] == misconception_id
        )

        return {
            "skill": skill,
            "misconception_id": misconception["id"],
            "misconception": misconception["name"],
            "confidence": 0.85,
            "evidence": traceback or "Code execution timed out."
        }

    return {
        "skill": skill,
        "misconception_id": misconception_id,
        "misconception": "Unknown misconception",
        "confidence": 0.40,
        "evidence": traceback or "Incorrect output."
    }

In [ ]:

# TEST 1: Infinite loop

test1 = {
    "passed": False,
    "error_type": "TIMEOUT",
    "output": "",
    "traceback": "Code execution exceeded the time limit."
}

print("TEST 1:")
print(diagnose_error(test1))


# TEST 2: Missing None check

test2 = {
    "passed": False,
    "error_type": "RUNTIME_ERROR",
    "output": "",
    "traceback": "AttributeError: 'NoneType' object has no attribute 'next'"
}

print("\nTEST 2:")
print(diagnose_error(test2))


# TEST 3: Incorrect logic

test3 = {
    "passed": False,
    "error_type": "ASSERTION_ERROR",
    "output": "",
    "traceback": "AssertionError"
}

print("\nTEST 3:")
print(diagnose_error(test3))


# TEST 4: Correct code

test4 = {
    "passed": True,
    "error_type": None,
    "output": "All tests passed!",
    "traceback": None
}

print("\nTEST 4:")
print(diagnose_error(test4))


TEST 1:
{'skill': 'linked_list_traversal', 'misconception_id': 'MC04', 'misconception': 'Possible infinite loop', 'confidence': 0.85, 'evidence': 'Code execution timed out. The pointer may not be advancing.'}

TEST 2:
{'skill': 'linked_list_traversal', 'misconception_id': 'MC02', 'misconception': 'Missing None check', 'confidence': 0.9, 'evidence': 'The code attempted to access an attribute of a None value.'}

TEST 3:
{'skill': 'linked_list_traversal', 'misconception_id': 'MC_UNKNOWN', 'misconception': 'Incorrect output or logic', 'confidence': 0.6, 'evidence': 'The actual output did not match the expected output.'}

TEST 4:
{'skill': 'linked_list_traversal', 'misconception_id': None, 'misconception': 'No identified misconception', 'confidence': 1.0, 'evidence': 'All tests passed.'}


####Integrating person A

In [1]:
from google.colab import files

uploaded = files.upload()

Saving domain_knowledge_base.md to domain_knowledge_base.md


In [3]:
import json

with open("domain_knowledge_base.md", "r", encoding="utf-8") as file:
    domain_data = json.load(file)

print("Domain Knowledge Base loaded successfully!")

print("Skills:", len(domain_data["skills"]))
print("Misconceptions:", len(domain_data["misconceptions"]))
print("Questions:", len(domain_data["questions"]))

Domain Knowledge Base loaded successfully!
Skills: 7
Misconceptions: 8
Questions: 12


In [4]:
for item in domain_data["misconceptions"]:
    print(item["id"], "-", item["name"])

MIS-01 - Losing Head Pointer
MIS-02 - Missing None Check
MIS-03 - Wrong Pointer Reassignment Order
MIS-04 - Infinite Loop
MIS-05 - Forgetting Head Update
MIS-06 - Off-by-One in Position Search
MIS-07 - Value vs Node Confusion
MIS-08 - Single Node Edge Case Failure


In [6]:
for item in domain_data["skills"]:
    print(item)

{'id': 'SKL-01', 'name': 'Node Creation & Basic Attributes', 'description': 'Defining the Node class with data and next pointer attributes.', 'prerequisites': []}
{'id': 'SKL-02', 'name': 'Traversal & Inspection', 'description': 'Iterating through the list using a temporary pointer without losing head reference.', 'prerequisites': ['SKL-01']}
{'id': 'SKL-03', 'name': 'Insertion at Head', 'description': 'Creating a new node, pointing its next to current head, and updating head.', 'prerequisites': ['SKL-01']}
{'id': 'SKL-04', 'name': 'Insertion at Tail/Middle', 'description': 'Navigating to a specific index/end and updating pointers in correct sequence.', 'prerequisites': ['SKL-02', 'SKL-03']}
{'id': 'SKL-05', 'name': 'Deletion of Node', 'description': 'Locating target/index node, updating preceding pointers, and handling memory references.', 'prerequisites': ['SKL-02', 'SKL-04']}
{'id': 'SKL-06', 'name': 'Null & Edge-Case Handling', 'description': 'Handling operations on empty lists (he

In [7]:
print(json.dumps(
    domain_data["misconceptions"][0],
    indent=4
))

{
    "id": "MIS-01",
    "name": "Losing Head Pointer",
    "root_cause": "Student mutates head directly during iteration instead of using a temporary variable.",
    "deterministic_trigger": "Head reassignment during loop / head becomes None unexpectedly",
    "hints": {
        "L1": "Take a look at how you are traversing the list. Are you keeping track of where the list starts?",
        "L2": "Modifying `head` directly moves the start pointer of your list. Use a temporary pointer like `curr = head` to iterate instead.",
        "L3": "Example:\ncurr = head\nwhile curr:\n    print(curr.val)\n    curr = curr.next  # head remains untouched!",
        "L4": "curr = head\nwhile curr:\n    # Process curr.val\n    curr = ___ # Update loop variable, NOT head"
    }
}


In [8]:
for item in domain_data["misconceptions"]:

    print("\n" + "=" * 50)
    print("ID:", item["id"])
    print("Name:", item["name"])
    print("Root Cause:", item["root_cause"])
    print("Trigger:", item["deterministic_trigger"])

    print("\nHints:")
    for level, hint in item["hints"].items():
        print(level, ":", hint)


ID: MIS-01
Name: Losing Head Pointer
Root Cause: Student mutates head directly during iteration instead of using a temporary variable.
Trigger: Head reassignment during loop / head becomes None unexpectedly

Hints:
L1 : Take a look at how you are traversing the list. Are you keeping track of where the list starts?
L2 : Modifying `head` directly moves the start pointer of your list. Use a temporary pointer like `curr = head` to iterate instead.
L3 : Example:
curr = head
while curr:
    print(curr.val)
    curr = curr.next  # head remains untouched!
L4 : curr = head
while curr:
    # Process curr.val
    curr = ___ # Update loop variable, NOT head

ID: MIS-02
Name: Missing None Check
Root Cause: Accessing attributes on a pointer variable that evaluates to None.
Trigger: AttributeError: 'NoneType' object has no attribute

Hints:
L1 : What happens if your node pointer becomes `None` before you access `.next` or `.val`?
L2 : You are attempting to access properties of a node that is `None`.

###Step-3

In [19]:
from google import genai
import google.auth

print("Gemini SDK imported successfully!")
print("Google Auth version:", google.auth.__version__)

Gemini SDK imported successfully!
Google Auth version: 2.49.0


In [20]:
from getpass import getpass

api_key = getpass("Enter your Gemini api key:")

print("API key received successfully!")

Enter your Gemini api key:··········
API key received successfully!


In [21]:
from google import genai

client = genai.Client(api_key=api_key)

MODEL_NAME = "gemini-3.6-flash"

response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Explain what a linked list is in two simple sentences."
)

print("Gemini connection successful!")
print(response.text)

Gemini connection successful!
A linked list is a linear chain of data items, called nodes, where each node contains a piece of information and a link pointing to the next node. This structure allows you to easily add or remove items anywhere in the list without having to rearrange the rest of the data.


###Step-4


In [25]:
import json
from google.genai import types

In [33]:
def diagnose_with_gemini(student_code, runner_result, skill="SKL-02"):

    # Prepare Person A's official misconception catalogue
    catalogue = []

    for item in domain_data["misconceptions"]:
        catalogue.append({
            "id": item["id"],
            "name": item["name"],
            "root_cause": item["root_cause"],
            "deterministic_trigger": item["deterministic_trigger"]
        })

    prompt = f"""
    You are an expert Python and Data Structures tutor.

    Identify the most likely misconception in the student's code.

    Student skill: {skill}

    Student code:
    {student_code}

    Execution result:
    {json.dumps(runner_result)}

    OFFICIAL MISCONCEPTION CATALOGUE:
    {json.dumps(catalogue, indent=2)}

    Instructions:
1. Analyse the student's code and execution result carefully.
2. Compare the mistake with every official misconception.
3. Select an ID ONLY when the root cause clearly matches.
4. Do not select a misconception just because it sounds similar.
5. MIS-06 is specifically about incorrect position indexing
   during insertion or deletion, NOT skipping nodes during traversal.
6. If no official misconception accurately matches, return MC_UNKNOWN.
7. Never invent an ID.
8. Do not provide the complete corrected solution.

If the student uses current.next.next during normal
linked-list traversal, check whether any official
misconception explicitly covers skipping nodes.
If none does, return MC_UNKNOWN.
    Return JSON with these fields:
    {{
        "skill": "{skill}",
        "misconception_id": "MIS-01",
        "misconception": "Misconception name",
        "confidence": 0.0,
        "evidence": "Evidence from the student's code"
    }}
    """

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            )
        )

        diagnosis = json.loads(response.text)

        allowed_ids = {
            item["id"] for item in domain_data["misconceptions"]
        }

        required_keys = [
            "skill",
            "misconception_id",
            "misconception",
            "confidence",
            "evidence"
        ]

        for key in required_keys:
            if key not in diagnosis:
                raise ValueError(f"Missing field: {key}")

        if diagnosis["misconception_id"] not in allowed_ids | {"MC_UNKNOWN"}:
            raise ValueError("Gemini returned an invalid misconception ID")

        confidence = float(diagnosis["confidence"])

        if not 0 <= confidence <= 1:
            raise ValueError("Invalid confidence value")

        diagnosis["confidence"] = confidence

        # Replace the name with the official catalogue name
        if diagnosis["misconception_id"] in allowed_ids:

            official_item = next(
                item for item in domain_data["misconceptions"]
                if item["id"] == diagnosis["misconception_id"]
            )

            diagnosis["misconception"] = official_item["name"]

        return diagnosis

    except Exception as e:
        return {
            "skill": skill,
            "misconception_id": "MC_UNKNOWN",
            "misconception": "Diagnosis unavailable",
            "confidence": 0.0,
            "evidence": str(e)
        }

In [27]:
student_code = """
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

head = Node(10)

while head is not None:
    print(head.data)
"""

test_result = {
    "passed": False,
    "error_type": "TIMEOUT",
    "output": "",
    "traceback": "Code execution exceeded the time limit."
}

diagnosis = diagnose_with_gemini(student_code, test_result)

print(json.dumps(diagnosis, indent=4))

{
    "skill": "SKL-02",
    "misconception_id": "MIS-04",
    "misconception": "Infinite Loop",
    "confidence": 0.98,
    "evidence": "Inside the `while head is not None:` loop, the `head` pointer is never advanced to `head.next`, causing the loop to run indefinitely and result in a timeout error."
}


###Step-5

In [34]:
def diagnose_student_submission(student_code, runner_result, skill="SKL-02"):

    # First, try rule-based diagnosis
    rule_result = diagnose_error(runner_result, skill)

    # If rules identify a known misconception, return it
    if rule_result["misconception_id"] not in [None, "MC_UNKNOWN"]:

        rule_result["diagnosis_source"] = "rule_based"
        return rule_result

    # If code passes all tests, no LLM diagnosis is needed
    if runner_result.get("passed"):
        rule_result["diagnosis_source"] = "rule_based"
        return rule_result

    # Otherwise, ask Gemini
    gemini_result = diagnose_with_gemini(
        student_code,
        runner_result,
        skill
    )

    gemini_result["diagnosis_source"] = "gemini"

    return gemini_result

####test

In [29]:
student_code = """
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

head = Node(10)

while head is not None:
    print(head.data)
"""

test_result = {
    "passed": False,
    "error_type": "TIMEOUT",
    "output": "",
    "traceback": "Code execution exceeded the time limit."
}

final_diagnosis = diagnose_student_submission(
    student_code,
    test_result
)

print(json.dumps(final_diagnosis, indent=4))

{
    "skill": "linked_list_traversal",
    "misconception_id": "MIS-04",
    "misconception": "Infinite Loop",
    "confidence": 0.85,
    "evidence": "Code execution exceeded the time limit.",
    "diagnosis_source": "rule_based"
}


In [35]:
student_code = """
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

head = Node(10)
head.next = Node(20)
head.next.next = Node(30)

current = head
values = []

while current is not None:
    values.append(current.data)
    current = current.next.next

print(values)
"""

test_result = {
    "passed": False,
    "error_type": "ASSERTION_ERROR",
    "output": "[10, 30]",
    "traceback": "AssertionError: Expected [10, 20, 30], got [10, 30]"
}

final_diagnosis = diagnose_student_submission(
    student_code,
    test_result
)

print(json.dumps(final_diagnosis, indent=4))

{
    "skill": "SKL-02",
    "misconception_id": "MC_UNKNOWN",
    "misconception": "Unknown Misconception",
    "confidence": 1.0,
    "evidence": "The student reassigns `current = current.next.next` instead of `current = current.next` during traversal, causing every alternate node to be skipped. None of the official misconceptions explicitly cover node-skipping during traversal.",
    "diagnosis_source": "gemini"
}


###Step-6 Personalized hint generator

In [ ]:
def generate_hint(student_code, diagnosis, hint_level, skill="SKL-02"):

    hint_instructions = {
        "L1": "Give only one guiding question. Do not explain the answer.",
        "L2": "Give a conceptual explanation without writing corrected code.",
        "L3": "Give a similar but different worked example. Do not solve the student's exact problem.",
        "L4": "Give a partial solution with meaningful blanks. Never provide the complete solution."
    }

    if hint_level not in hint_instructions:
        raise ValueError("Hint level must be L1, L2, L3, or L4")

    prompt = f"""
    You are a friendly personal Data Structures tutor.

    Student skill: {skill}

    Student's code:
    {student_code}

    Diagnosis:
    {json.dumps(diagnosis)}

    Requested hint level: {hint_level}

    Instructions:
    {hint_instructions[hint_level]}

    Rules:
    1. Focus on the identified misconception.
    2. Use simple beginner-friendly language.
    3. Never reveal the complete solution.
    4. Do not give the answer directly at L1 or L2.
    5. At L3, use a different example.
    6. At L4, leave meaningful blanks in the code.
    7. Keep the hint short and useful.

    Return JSON in this format:

    {{
        "hint_level": "{hint_level}",
        "hint": "Your personalised hint"
    }}
    """

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            )
        )

        hint_result = json.loads(response.text)

        if hint_result.get("hint_level") != hint_level:
            raise ValueError("Incorrect hint level returned")

        if not isinstance(hint_result.get("hint"), str):
            raise ValueError("Invalid hint format")

        if not hint_result["hint"].strip():
            raise ValueError("Empty hint returned")

        return hint_result

    except Exception as e:
        return {
            "hint_level": hint_level,
            "hint": "Hint generation is temporarily unavailable. Please try again.",
            "error": str(e)
        }

In [ ]:
student_code = """
current = head

while current is not None:
    values.append(current.data)
    current = current.next.next
"""

for level in ["L1", "L2", "L3", "L4"]:

    print("\n", "=" * 40)
    print("HINT LEVEL:", level)
    print("=" * 40)

    hint = generate_hint(
        student_code,
        final_diagnosis,
        level
    )

    print(hint["hint"])


HINT LEVEL: L1
How many nodes forward does your pointer move during each step of the loop?

HINT LEVEL: L2
Think about how you move step-by-step through a linked list. Each time you access `.next`, you move forward by one node. What happens to the nodes in your list if you chain more than one `.next` together in a single step?

HINT LEVEL: L3
Here is a similar example showing how to process every single node in a list without skipping any:

If we want to print all numbers in the list `5 -> 15 -> 25`:

```python
curr = head
while curr is not None:
    print(curr.data)
    curr = curr.next  # Moves to the immediate next node
```

Notice how using `curr = curr.next` moves forward by just one step each time, ensuring no nodes (like `15`) get skipped!

HINT LEVEL: L4
To visit every node without skipping any, you need to move your `current` pointer just one step forward during each iteration.

Fill in the blank to complete the loop:
```python
while current is not None:
    values.append(cur

####Person-A integration

In [38]:
def generate_catalogue_hint(misconception_id, hint_level):

    # Find the matching misconception in Person A's catalogue
    misconception = next(
        (
            item for item in domain_data["misconceptions"]
            if item["id"] == misconception_id
        ),
        None
    )

    # Check whether the misconception exists
    if misconception is None:
        return "Misconception not found."

    # Check whether the requested hint level exists
    if hint_level not in misconception["hints"]:
        return "Hint level not available."

    # Return Person A's official hint
    return misconception["hints"][hint_level]

In [39]:
print(generate_catalogue_hint("MIS-04", "L1"))
print(generate_catalogue_hint("MIS-04", "L2"))
print(generate_catalogue_hint("MIS-04", "L3"))
print(generate_catalogue_hint("MIS-04", "L4"))

Your loop isn't terminating. Is your pointer moving forward with each step?
A `while` loop over a linked list requires advancing the current reference at the end of each iteration.
Example:
while curr:
    # do work
    curr = curr.next # Essential increment step!
while curr:
    # do work
    ___ = curr.next


In [40]:
def get_personalized_hint(student_code, diagnosis, hint_level):

    misconception_id = diagnosis["misconception_id"]

    # Check whether the diagnosis exists in Person A's catalogue
    official_ids = {
        item["id"] for item in domain_data["misconceptions"]
    }

    if misconception_id in official_ids:

        # Retrieve the official hint
        hint = generate_catalogue_hint(
            misconception_id,
            hint_level
        )

        return {
            "hint_level": hint_level,
            "hint": hint,
            "hint_source": "person_a_catalogue"
        }

    # If the misconception is unknown, use Gemini
    else:

        hint_result = generate_hint(
            student_code,
            diagnosis,
            hint_level
        )

        hint_result["hint_source"] = "gemini"
        return hint_result

In [41]:
test_diagnosis = {
    "skill": "SKL-02",
    "misconception_id": "MIS-04",
    "misconception": "Infinite Loop",
    "confidence": 0.85
}

hint_result = get_personalized_hint(
    student_code,
    test_diagnosis,
    "L1"
)

print(json.dumps(hint_result, indent=4))

{
    "hint_level": "L1",
    "hint": "Your loop isn't terminating. Is your pointer moving forward with each step?",
    "hint_source": "person_a_catalogue"
}
